# Part 1: Data Preparation
 This part loads the raw CSV, cleans/translates it, be used in part 2 (visuals) and 3 (training)

## Section A: import, translation, cleaning

In [ ]:
import os
import json
from typing import Final
import numpy as np
import pandas as pd

# --- Reproducibility ---
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

df = pd.read_csv('mexico_covid19.csv')
translation_dict = {
    "FECHA_ARCHIVO": "report_date",
    "ID_REGISTRO": "registry_id",
    "ENTIDAD_UM": "medical_unit_state",
    "ENTIDAD_RES": "residence_state",
    "RESULTADO": "result", #RT-PCR test result
    "DELAY": "days_to_confirmation",
    "ENTIDAD_REGISTRO": "registration_state",
    "ENTIDAD": "state_name",
    "ABR_ENT": "state_abbreviation",
    "FECHA_ACTUALIZACION": "update_date",
    "ORIGEN": "origin",
    "SECTOR": "sector",
    "SEXO": "sex",
    "ENTIDAD_NAC": "birth_state",
    "MUNICIPIO_RES": "residence_municipality",
    "TIPO_PACIENTE": "patient_type",
    "FECHA_INGRESO": "admission_date",
    "FECHA_SINTOMAS": "symptoms_onset_date",
    "FECHA_DEF": "death_date",
    "INTUBADO": "intubated",
    "NEUMONIA": "pneumonia",
    "EDAD": "age",
    "NACIONALIDAD": "nationality",
    "EMBARAZO": "pregnant",
    "HABLA_LENGUA_INDIG": "speaks_indigenous_language",
    "DIABETES": "diabetes",
    "EPOC": "COPD",
    "ASMA": "asthma",
    "INMUSUPR": "immunosuppressed",
    "HIPERTENSION": "hypertension",
    "OTRA_COM": "other_comorbidity",
    "CARDIOVASCULAR": "cardiovascular",
    "OBESIDAD": "obesity",
    "RENAL_CRONICA": "chronic_renal_failure",
    "TABAQUISMO": "tobacco_use",
    "OTRO_CASO": "contact_with_other_case",
    "MIGRANTE": "migrant",
    "PAIS_NACIONALIDAD": "nationality_country",
    "PAIS_ORIGEN": "origin_country",
    "UCI": "ICU",
}
df = df.rename(columns=translation_dict)
df

,id,report_date,registry_id,medical_unit_state,residence_state,result,days_to_confirmation,registration_state,state_name,state_abbreviation,...,other_comorbidity,cardiovascular,obesity,chronic_renal_failure,tobacco_use,contact_with_other_case,migrant,nationality_country,origin_country,ICU
0,9269,2020-04-12,00011f,25,25,2,0,25,Sinaloa,SL,...,2,2,1,2,2,2,99,MÃ©xico,97,97
1,33333,2020-04-12,00014e,14,14,2,0,14,Jalisco,JC,...,2,2,1,2,1,99,99,MÃ©xico,97,2
2,35483,2020-04-12,000153,8,8,1,0,8,Chihuahua,CH,...,2,2,2,2,2,99,99,MÃ©xico,97,2
3,7062,2020-04-12,0001b6,9,15,1,0,9,Ciudad de Mexico,DF,...,2,2,1,2,2,99,99,MÃ©xico,97,97
4,23745,2020-04-12,0001c1,9,9,2,0,9,Ciudad de Mexico,DF,...,2,2,2,2,2,99,99,MÃ©xico,97,97
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
263002,7094887,2020-06-03,1e019c,32,32,1,0,32,Zacatecas,ZS,...,2,2,2,2,2,1,99,MÃ©xico,99,97
263003,7053721,2020-06-03,1e2b05,12,12,1,0,12,Guerrero,GR,...,2,2,1,2,2,99,99,MÃ©xico,99,2
263004,7055429,2020-06-03,1e473f,20,20,1,0,20,Oaxaca,OC,...,2,2,2,1,2,99,99,MÃ©xico,99,2
263005,7043768,2020-06-03,1e6da1,13,13,1,0,13,Hidalgo,HG,...,2,2,2,2,2,2,99,MÃ©xico,99,2


In [ ]:
# We don't translate municipios, we want a well generalized model which which does a prediction based on medicial information of the patient, not on where he/she lives

# YES/NO fields: 1=Yes, 2=No, 97=N/A, 98=Ignored, 99=Not specified — this comes from the SI_NO catalog
yes_no_map = {1: 'Yes', 2: 'No', 97: 'N/A', 98: 'Ignored', 99: 'Not specified'}
yes_no_cols = [
    'intubated', 'pneumonia', 'pregnant', 'speaks_indigenous_language',
    'diabetes', 'COPD', 'asthma', 'immunosuppressed', 'hypertension',
    'other_comorbidity', 'cardiovascular', 'obesity', 'chronic_renal_failure',
    'tobacco_use', 'contact_with_other_case', 'migrant', 'ICU',
]
for col in yes_no_cols:
    df[col] = df[col].map(yes_no_map)

# origin — this comes from the ORIGEN catalog
df['origin'] = df['origin'].map({1: 'USMER', 2: 'Outside USMER', 99: 'Not specified'})

# sector — this comes from the SECTOR catalog
df['sector'] = df['sector'].map({
    1: 'Red Cross', 2: 'DIF', 3: 'State', 4: 'IMSS', 5: 'IMSS-Bienestar',
    6: 'ISSSTE', 7: 'Municipal', 8: 'PEMEX', 9: 'Private',
    10: 'SEDENA (Army)', 11: 'SEMAR (Navy)', 12: 'SSA (Ministry of Health)',
    13: 'University', 99: 'Not specified',
})

# sex — SEXO catalog (1=Female, 2=Male),99 doesnt exist in the df
n_sex_99 = (df['sex'] == 99).sum()
print(f"{n_sex_99} rows with sex = 99 (not specified)")
df['sex'] = df['sex'].map({1: 0, 2: 1})


# patient_type — this comes from the TIPO_PACIENTE catalog
n_pat_99 = (df['patient_type'] == 99).sum()
print(f"{n_pat_99} rows with patient_type = 99 (not specified)")
df['patient_type'] = df['patient_type'].map({1: 'Outpatient', 2: 'Hospitalized'})

# nationality — NACIONALIDAD catalog (1=Mexican, 2=Foreign), 99 doesnt exist in the df
n_nat_99 = (df['nationality'] == 99).sum()
print(f"{n_nat_99} rows with nationality = 99 (not specified)")
df['nationality'] = df['nationality'].map({1: 0, 2: 1})


# result — this comes from the RESULTADO catalog
df['result'] = df['result'].map({
    1: 'Positive', 2: 'Negative'
})

# Mexican state codes (ENTIDAD_UM, ENTIDAD_RES, ENTIDAD_NAC, ENTIDAD_REGISTRO) — this comes from the ENTIDADES catalog
state_map = {
    1: 'Aguascalientes', 2: 'Baja California', 3: 'Baja California Sur',
    4: 'Campeche', 5: 'Coahuila', 6: 'Colima', 7: 'Chiapas', 8: 'Chihuahua',
    9: 'Ciudad de México', 10: 'Durango', 11: 'Guanajuato', 12: 'Guerrero',
    13: 'Hidalgo', 14: 'Jalisco', 15: 'Estado de México', 16: 'Michoacán',
    17: 'Morelos', 18: 'Nayarit', 19: 'Nuevo León', 20: 'Oaxaca',
    21: 'Puebla', 22: 'Querétaro', 23: 'Quintana Roo', 24: 'San Luis Potosí',
    25: 'Sinaloa', 26: 'Sonora', 27: 'Tabasco', 28: 'Tamaulipas',
    29: 'Tlaxcala', 30: 'Veracruz', 31: 'Yucatán', 32: 'Zacatecas', 36:'Mexico',
    97: 'N/A', 98: 'Ignored', 99: 'Not specified',
}
for col in ['medical_unit_state', 'residence_state', 'birth_state', 'registration_state']:
    df[col] = df[col].map(state_map)

# Date columns — replace death_date sentinel then convert all to datetime
df['death_date'] = df['death_date'].replace('9999-99-99', None)

date_cols = ['report_date', 'update_date', 'admission_date', 'symptoms_onset_date', 'death_date']
for col in date_cols:
    df[col] = pd.to_datetime(df[col], format='%Y-%m-%d', errors='coerce')

df

0 rows with sex = 99 (not specified)
0 rows with patient_type = 99 (not specified)
0 rows with nationality = 99 (not specified)


,id,report_date,registry_id,medical_unit_state,residence_state,result,days_to_confirmation,registration_state,state_name,state_abbreviation,...,other_comorbidity,cardiovascular,obesity,chronic_renal_failure,tobacco_use,contact_with_other_case,migrant,nationality_country,origin_country,ICU
0,9269,2020-04-12,00011f,Sinaloa,Sinaloa,Negative,0,Sinaloa,Sinaloa,SL,...,No,No,Yes,No,No,No,Not specified,MÃ©xico,97,N/A
1,33333,2020-04-12,00014e,Jalisco,Jalisco,Negative,0,Jalisco,Jalisco,JC,...,No,No,Yes,No,Yes,Not specified,Not specified,MÃ©xico,97,No
2,35483,2020-04-12,000153,Chihuahua,Chihuahua,Positive,0,Chihuahua,Chihuahua,CH,...,No,No,No,No,No,Not specified,Not specified,MÃ©xico,97,No
3,7062,2020-04-12,0001b6,Ciudad de México,Estado de México,Positive,0,Ciudad de México,Ciudad de Mexico,DF,...,No,No,Yes,No,No,Not specified,Not specified,MÃ©xico,97,N/A
4,23745,2020-04-12,0001c1,Ciudad de México,Ciudad de México,Negative,0,Ciudad de México,Ciudad de Mexico,DF,...,No,No,No,No,No,Not specified,Not specified,MÃ©xico,97,N/A
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
263002,7094887,2020-06-03,1e019c,Zacatecas,Zacatecas,Positive,0,Zacatecas,Zacatecas,ZS,...,No,No,No,No,No,Yes,Not specified,MÃ©xico,99,N/A
263003,7053721,2020-06-03,1e2b05,Guerrero,Guerrero,Positive,0,Guerrero,Guerrero,GR,...,No,No,Yes,No,No,Not specified,Not specified,MÃ©xico,99,No
263004,7055429,2020-06-03,1e473f,Oaxaca,Oaxaca,Positive,0,Oaxaca,Oaxaca,OC,...,No,No,No,Yes,No,Not specified,Not specified,MÃ©xico,99,No
263005,7043768,2020-06-03,1e6da1,Hidalgo,Hidalgo,Positive,0,Hidalgo,Hidalgo,HG,...,No,No,No,No,No,No,Not specified,MÃ©xico,99,No


In [ ]:
# check target distribution: hospitalized (1) vs outpatient (0)
df = df[df['patient_type'].isin(['Outpatient', 'Hospitalized'])].copy()
df['target'] = (df['patient_type'] == 'Hospitalized').astype(int)

print("\nClass distribution (0 = Outpatient, 1 = Hospitalized):")
print(df['target'].value_counts().sort_index())
print(f"Hospitalization prevalence: {df['target'].mean():.3f}")



Class distribution (0 = Outpatient, 1 = Hospitalized):
target
0    200838
1     62169
Name: count, dtype: int64
Hospitalization prevalence: 0.236


## Section B: Feature selection 

In [4]:

numeric_features = ['age']
comorbidity_cols = [
    'diabetes', 'COPD', 'asthma', 'immunosuppressed', 'hypertension',
    'other_comorbidity', 'cardiovascular', 'obesity', 'chronic_renal_failure',
    'tobacco_use',
]

binary_features = comorbidity_cols + ['pregnant', 'treated_in_residence_state', 'sex', 'nationality']

categorical_features = ['contact_with_other_case']

visualization_df = df.copy()

visualization_df['treated_in_residence_state'] = np.where(
    visualization_df['medical_unit_state'] == visualization_df['residence_state'], 'Yes', 'No')


visualization_df['died'] = visualization_df['death_date'].notna().astype(int)

binary_map = {'Yes': 1, 'No': 0}
for col in comorbidity_cols + ['pregnant', 'treated_in_residence_state']:
    visualization_df[col] = visualization_df[col].map(binary_map)

# we just say if we don't have information for pregnant we just say no
for col in ['pregnant']:
    visualization_df[col] = visualization_df[col].fillna(0)

# if we have no information we keep unkown as third category
visualization_df['contact_with_other_case'] = visualization_df['contact_with_other_case'].where(
    visualization_df['contact_with_other_case'].isin(['Yes', 'No']), 'Unknown')

visualization_df['N_COMORBIDITIES'] = visualization_df[comorbidity_cols].sum(axis=1).astype(int)
numeric_features = numeric_features + ['N_COMORBIDITIES']



# for the remaining columns if there is a null value we just drop the rows we dont lose a lot
drop_subset = comorbidity_cols + numeric_features
rows_before = len(visualization_df)
missing_per_col = visualization_df[drop_subset].isna().sum()
visualization_df = visualization_df.dropna(subset=drop_subset).reset_index(drop=True)
rows_after = len(visualization_df)
print(f"rows before dropping: {rows_before}")
print(f"rows after dropping: {rows_after}")
print(f"precentage dropped: {100*(1- rows_after/rows_before)}%")

rows before dropping: 263007
rows after dropping: 260933
precentage dropped: 0.7885721672807211%


In [5]:
# training_df leakage safe only usable features
feature_cols = numeric_features + binary_features + categorical_features

training_df = visualization_df[feature_cols + ['target']].copy()

X = training_df[feature_cols]
y = training_df['target']

print(f"Design matrix: {X.shape[0]:,} rows x {X.shape[1]} features")
print(f"Feature columns: {feature_cols}")

Design matrix: 260,933 rows x 17 features
Feature columns: ['age', 'N_COMORBIDITIES', 'diabetes', 'COPD', 'asthma', 'immunosuppressed', 'hypertension', 'other_comorbidity', 'cardiovascular', 'obesity', 'chronic_renal_failure', 'tobacco_use', 'pregnant', 'treated_in_residence_state', 'sex', 'nationality', 'contact_with_other_case']


In [ ]:
# Persist prepared objects for the visuals & training notebooks
os.makedirs('artifacts', exist_ok=True)
visualization_df.to_pickle('artifacts/visualization_df.pkl')   # cleaned frame, all columns (part 2 visuals)
training_df.to_pickle('artifacts/training_df.pkl')             # leakage-safe modelling frame (part 3 training)

with open('artifacts/feature_lists.json', 'w') as f:
    json.dump({
        'numeric_features': numeric_features,
        'binary_features': binary_features,
        'categorical_features': categorical_features,
        'comorbidity_cols': comorbidity_cols,
        'feature_cols': feature_cols,
    }, f, indent=2)

print("Saved to artifacts/: visualization_df.pkl, training_df.pkl, feature_lists.json")

Saved to artifacts/: visualization_df.pkl, training_df.pkl, feature_lists.json
